In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
dataset_path = "/content/drive/MyDrive/Skin Disease"


In [4]:
import os

print(os.listdir(dataset_path))


['10. Warts Molluscum and other Viral Infections - 2103', '6. Benign Keratosis-like Lesions (BKL) 2624', '4. Basal Cell Carcinoma (BCC) 3323', '8. Seborrheic Keratoses and other Benign Tumors - 1.8k', '9. Tinea Ringworm Candidiasis and other Fungal Infections - 1.7k', '3. Atopic Dermatitis - 1.25k', '5. Melanocytic Nevi (NV) - 7970', '1. Eczema 1677', '7. Psoriasis pictures Lichen Planus and related diseases - 2k', '2. Melanoma 15.75k', 'train', 'val', 'test']


In [5]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import load_img, img_to_array   # ✅ use these instead
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model


In [6]:
import os, shutil
from sklearn.model_selection import train_test_split


In [7]:
classes = os.listdir(dataset_path)

for cls in classes:
    cls_path = os.path.join(dataset_path, cls)
    if not os.path.isdir(cls_path):
        continue

    images = [img for img in os.listdir(cls_path) if img.lower().endswith((".jpg",".jpeg",".png"))]
    print(f"{cls}: {len(images)} images")



10. Warts Molluscum and other Viral Infections - 2103: 173 images
6. Benign Keratosis-like Lesions (BKL) 2624: 1023 images
4. Basal Cell Carcinoma (BCC) 3323: 1540 images
8. Seborrheic Keratoses and other Benign Tumors - 1.8k: 868 images
9. Tinea Ringworm Candidiasis and other Fungal Infections - 1.7k: 1702 images
3. Atopic Dermatitis - 1.25k: 1030 images
5. Melanocytic Nevi (NV) - 7970: 2477 images
1. Eczema 1677: 1033 images
7. Psoriasis pictures Lichen Planus and related diseases - 2k: 2055 images
2. Melanoma 15.75k: 1016 images
train: 0 images
val: 0 images
test: 0 images


In [8]:
for cls in classes:
    cls_path = os.path.join(dataset_path, cls)
    if not os.path.isdir(cls_path):
        continue

    images = [img for img in os.listdir(cls_path) if img.lower().endswith((".jpg",".jpeg",".png"))]

    if len(images) == 0:
        print(f"⚠️ Skipping {cls} (no images found)")
        continue


⚠️ Skipping train (no images found)
⚠️ Skipping val (no images found)
⚠️ Skipping test (no images found)


In [9]:
classes = [cls for cls in os.listdir(dataset_path) if cls not in ["train", "val", "test"]]


In [10]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=20,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    dataset_path + "/train",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)

val_generator = val_datagen.flow_from_directory(
    dataset_path + "/val",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)

test_generator = test_datagen.flow_from_directory(
    dataset_path + "/test",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)


Found 10328 images belonging to 10 classes.
Found 2844 images belonging to 10 classes.
Found 2972 images belonging to 10 classes.


In [11]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=20,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    dataset_path + "/train",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)

val_generator = val_datagen.flow_from_directory(
    dataset_path + "/val",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)

test_generator = test_datagen.flow_from_directory(
    dataset_path + "/test",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)


Found 10328 images belonging to 10 classes.
Found 2844 images belonging to 10 classes.
Found 2972 images belonging to 10 classes.


In [12]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

# Load EfficientNetB0 base model
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))

# Freeze base layers initially
for layer in base_model.layers:
    layer.trainable = False

# Add custom layers
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(1024, activation='relu')(x)
output = Dense(train_generator.num_classes, activation='softmax')(x)

# Define final model
model = Model(inputs=base_model.input, outputs=output)

# Compile model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_2         │ (None, 224, 224,  │          0 │ input_layer_1[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_1     │ (None, 224, 224,  │          7 │ rescaling_2[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_3         │ (None, 224, 224,  │          0 │ normalization_1[… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_3[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 5,371,565 (20.49 MB)

 Trainable params: 1,321,994 (5.04 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [13]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',  # one-hot labels
    metrics=['accuracy']
)


In [ ]:
import os
from PIL import Image

def clean_dataset(dataset_path):
    bad_files = []
    for root, dirs, files in os.walk(dataset_path):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                # Try opening the image
                img = Image.open(file_path)
                img.verify()  # Verify integrity
            except Exception as e:
                print(f"Corrupted file removed: {file_path}")
                bad_files.append(file_path)
                os.remove(file_path)  # Delete bad file
    print(f"\n✅ Cleaning complete. {len(bad_files)} bad files removed.")

# Example usage
clean_dataset(dataset_path + "/train")
clean_dataset(dataset_path + "/val")
clean_dataset(dataset_path + "/test")


In [ ]:
history = model.fit(
    train_generator,              # your training dataset
    epochs=20,                    # start with 20, adjust later
    validation_data=val_generator # your validation dataset
)

Epoch 1/20
323/323 ━━━━━━━━━━━━━━━━━━━━ 6072s 19s/step - accuracy: 0.1769 - loss: 2.1985 - val_accuracy: 0.1751 - val_loss: 2.2103
Epoch 2/20
323/323 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.1901 - loss: 2.1854

In [ ]:
def clean_dataset(dataset_path):
    bad_files = []
    count = 0
    for root, dirs, files in os.walk(dataset_path):
        for file in files:
            file_path = os.path.join(root, file)
            count += 1
            if count % 500 == 0:
                print(f"Checked {count} files...")
            try:
                img = Image.open(file_path)
                img.verify()
            except Exception:
                print(f"Corrupted file removed: {file_path}")
                bad_files.append(file_path)
                os.remove(file_path)
    print(f"\n✅ Cleaning complete. {len(bad_files)} bad files removed.")


In [ ]:
history = model.fit(
    train_ds,
    epochs=50,
    validation_data=val_ds,
    callbacks=[early_stop]
)


In [ ]:
plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.legend()
plt.show()

plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.legend()
plt.show()


In [ ]:
print("Number of classes:", train_generator.num_classes)
print("Class names:", train_generator.class_indices)

print("\nTraining images:", train_generator.samples)
print("Validation images:", val_generator.samples)

print("\nModel output shape:", model.output_shape)

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam   # <-- Import Adam here

# Load EfficientNetB0 base model
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))

# Freeze base layers initially
for layer in base_model.layers:
    layer.trainable = False

# Add custom layers
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(1024, activation='relu')(x)
x = Dropout(0.5)(x)   # helps reduce overfitting
output = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

# Compile model (initial training)
model.compile(optimizer=Adam(learning_rate=1e-3),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Train frozen model first
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator
)

# 🔹 Fine‑tuning step
# Unfreeze last 50 layers for fine-tuning
for layer in base_model.layers[-50:]:
    layer.trainable = True

# Recompile with smaller learning rate
model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Fine-tune model
history_finetune = model.fit(
    train_generator,
    epochs=20,   # train longer
    validation_data=val_generator
)

# Evaluate on test set
test_loss, test_acc = model.evaluate(test_generator)
print("Test Accuracy:", test_acc)
print("Test Loss:", test_loss)

# Save model
model.save("skin_disease_detector_finetuned.h5")
